# Обучение нейронной сети с BPR-функцией потерь

## Описание
Данный ноутбук выполняет:
1. Загрузку предобработанной матрицы user-item
2. Обучение нейронной сети с BPR (Bayesian Personalized Ranking) на PyTorch
3. Извлечение эмбеддингов пользователей и товаров
4. Сохранение эмбеддингов и параметров обучения

## Архитектура модели FastBPR

```python
class FastBPR(nn.Module):
    """
    Нейронная сеть с BPR-функцией потерь для рекомендаций.
    
    Attributes:
        user_emb (nn.Embedding): Эмбеддинги пользователей [num_users x embedding_dim]
        item_emb (nn.Embedding): Эмбеддинги товаров [num_items x embedding_dim]
    
    Methods:
        forward(u: Tensor, i: Tensor, j: Tensor) -> tuple[Tensor, Tensor]:
            Прямой проход для BPR.
            
            Args:
                u: ID пользователей [batch_size]
                i: ID позитивных товаров [batch_size]
                j: ID негативных товаров [batch_size]
                
            Returns:
                tuple: (скоры для позитивных, скоры для негативных)
        
        get_embeddings() -> tuple[np.ndarray, np.ndarray]:
            Возвращает матрицы эмбеддингов пользователей и товаров.
    """

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
%cd /content/drive/MyDrive/VKR_TECD/
print("Текущая папка:", os.getcwd())

/content/drive/MyDrive/VKR_TECD
Текущая папка: /content/drive/MyDrive/VKR_TECD


In [ ]:
import numpy as np
from scipy.sparse import load_npz
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import time

print("ШАГ 3: ОБУЧЕНИЕ НЕЙРОННОЙ СЕТИ С BPR (УСКОРЕННАЯ ВЕРСИЯ)")

R = load_npz("output/train_matrix.npz")
print(f"Матрица: {R.shape[0]:,} x {R.shape[1]:,}")

num_users = R.shape[0]
num_items = R.shape[1]

# СУЩЕСТВЕННО УМЕНЬШЕННЫЕ ПАРАМЕТРЫ
embedding_dim = 16
lr = 0.01
epochs = 3
batch_size = 2048

print(f"embedding_dim: {embedding_dim}")
print(f"epochs: {epochs}")
print(f"batch_size: {batch_size}")

rows, cols = R.nonzero()
subset_size = 200000
rows = rows[:subset_size]
cols = cols[:subset_size]

rows_tensor = torch.tensor(rows, dtype=torch.long)
cols_tensor = torch.tensor(cols, dtype=torch.long)
edge_len = len(rows)
print(f"Взаимодействий: {edge_len:,}")

class FastBPR(nn.Module):
    def __init__(self, num_users, num_items, dim):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, dim)
        self.item_emb = nn.Embedding(num_items, dim)
        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)

    def forward(self, u, i, j):
        u_emb = self.user_emb(u)
        i_emb = self.item_emb(i)
        j_emb = self.item_emb(j)
        return (u_emb * i_emb).sum(dim=1), (u_emb * j_emb).sum(dim=1)

    def get_embeddings(self):
        return self.user_emb.weight.data.cpu().numpy(), self.item_emb.weight.data.cpu().numpy()

model = FastBPR(num_users, num_items, embedding_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

print("Обучение...")
start_time = time.time()

for epoch in range(epochs):
    total_loss = 0
    indices = np.random.permutation(edge_len)

    for i in range(0, edge_len, batch_size):
        batch_idx = indices[i:i+batch_size]
        u_idx = rows_tensor[batch_idx]
        pos_idx = cols_tensor[batch_idx]
        neg_idx = torch.randint(0, num_items, (len(batch_idx),))

        pos_score, neg_score = model(u_idx, pos_idx, neg_idx)
        loss = -F.logsigmoid(pos_score - neg_score).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"  Epoch {epoch+1}/{epochs}, Loss: {total_loss/(edge_len//batch_size):.4f}")

train_time = time.time() - start_time
print(f"Обучение завершено за {train_time:.2f} секунд")

user_embeddings, item_embeddings = model.get_embeddings()
np.save("output/user_embeddings_neural.npy", user_embeddings)
np.save("output/item_embeddings_neural.npy", item_embeddings)

params = {
    'model': 'Neural BPR (fast)',
    'embedding_dim': embedding_dim,
    'epochs': epochs,
    'train_time_sec': train_time
}
with open("output/neural_params.json", "w") as f:
    json.dump(params, f, indent=2)

print("Эмбеддинги сохранены!")

ШАГ 3: ОБУЧЕНИЕ НЕЙРОННОЙ СЕТИ С BPR (УСКОРЕННАЯ ВЕРСИЯ)
Матрица: 1,382,085 x 760,794
embedding_dim: 16
epochs: 3
batch_size: 2048
Взаимодействий: 200,000
Обучение...
  Epoch 1/3, Loss: 0.6755
  Epoch 2/3, Loss: 0.2721
  Epoch 3/3, Loss: 0.0714
Обучение завершено за 253.64 секунд
Эмбеддинги сохранены!
